In [2]:
import pandas as pd

person = pd.read_csv(r"C:\Users\Areeba\OneDrive\ECO346-A1\Data\pslm_hh_subset.csv")
household = pd.read_csv(r"C:\Users\Areeba\OneDrive\ECO346-A1\Data\pslm_person_subset.csv")

print("Person dataset shape:", person.shape)
print("Household dataset shape:", household.shape)

Person dataset shape: (19556, 109)
Household dataset shape: (107367, 47)


In [4]:
print(household.columns.tolist())

['hhid', 'idc', 'relation', 'sex', 'age', 'marital', 'born_here', 'dist_birth', 'move_year', 'dist_origin', 'mig_reason', 'diff_seeing', 'diff_hearing', 'diff_walking', 'diff_memory', 'diff_selfcare', 'diff_communicating', 'any_disability', 'can_read', 'can_write', 'can_math', 'school_type', 'grade_now', 'school_km', 'ed_admission', 'ed_uniform', 'ed_books', 'ed_exam', 'ed_tuition', 'ed_transport', 'ed_hostel', 'ed_other', 'ed_total', 'literate', 'emp_status', 'earn_month', 'months_worked', 'earn_year', 'emp_status2', 'earn_second', 'other_work', 'earn_other', 'sold_kind', 'earn_kind', 'inc_pension', 'income_shared_hh', 'earn_annual']


In [6]:
print(person.columns.tolist())

['hhid', 'psu', 'prov_code', 'urban_rural', 'dist_code', 'hhweight', 'hh_size', 'n_female', 'n_child', 'head_age', 'tenure', 'owner_sex', 'rooms', 'cook_fuel', 'light_fuel', 'water_source', 'water_dist_band', 'water_time_band', 'toilet_type', 'water_improved', 'toilet_flush', 'hh_inc_remit_dom', 'discordant_inc_remit_dom', 'hh_inc_remit_for', 'discordant_inc_remit_for', 'hh_inc_rent', 'discordant_inc_rent', 'hh_inc_other', 'discordant_inc_other', 'fies_worried', 'fies_healthy', 'fies_fewfoods', 'fies_skipped', 'fies_ateless', 'fies_ranout', 'fies_hungry', 'fies_wholeday', 'n_fies_answered', 'fies_raw', 'qty701', 'owns701', 'n_tv', 'has_tv', 'qty703', 'owns703', 'n_fridge', 'has_fridge', 'qty705', 'owns705', 'n_washer', 'has_washer', 'qty707', 'owns707', 'n_ac', 'has_ac', 'qty709', 'owns709', 'n_fan', 'has_fan', 'qty711', 'owns711', 'qty712', 'owns712', 'qty713', 'owns713', 'qty714', 'owns714', 'qty715', 'owns715', 'qty716', 'owns716', 'qty717', 'owns717', 'qty718', 'owns718', 'qty719',

In [7]:
combined = household.merge(
    person[["hhid", "hhweight", "prov_code", "urban_rural"]],
    on="hhid",
    how="inner"
)

print("Combined dataset shape:", combined.shape)

Combined dataset shape: (107367, 50)


In [8]:
# Select urban observations in KP
kp_urban = combined[
    (combined["prov_code"] == 1) &
    (combined["urban_rural"] == 2)
].copy()

# Keep earners aged 15–64 with positive annual earnings
earners = kp_urban[
    (kp_urban["age"] >= 15) &
    (kp_urban["age"] <= 64) &
    (kp_urban["earn_annual"] > 0)
].copy()

print("KP urban observations:", len(kp_urban))
print("Qualifying earners:", len(earners))

KP urban observations: 3450
Qualifying earners: 678


In [9]:
import numpy as np

# Create five-year age bands
age_labels = [
    "15-19", "20-24", "25-29", "30-34", "35-39",
    "40-44", "45-49", "50-54", "55-59", "60-64"
]

earners["age_band"] = pd.cut(
    earners["age"],
    bins=list(range(15, 66, 5)),
    labels=age_labels,
    right=False
)

print(earners[["age", "age_band"]].head(10))

     age age_band
531   38    35-39
536   50    50-54
537   46    45-49
541   45    45-49
544   17    15-19
546   55    55-59
548   22    20-24
556   27    25-29
559   40    40-44
566   42    40-44


In [10]:
# Calculate weighted mean earnings for each age band
band_table = (
    earners.groupby("age_band", observed=False)
    .apply(
        lambda group: np.average(
            group["earn_annual"],
            weights=group["hhweight"]
        ) if len(group) > 0 else np.nan
    )
    .reset_index(name="weighted_mean")
)

# Count observations in each age band
counts = (
    earners.groupby("age_band", observed=False)
    .size()
    .reset_index(name="n")
)

# Combine means and counts
band_table = band_table.merge(counts, on="age_band")

# Round weighted means to whole rupees
band_table["weighted_mean"] = band_table["weighted_mean"].round().astype(int)

print(band_table.to_string(index=False))

age_band  weighted_mean   n
   15-19         152102  35
   20-24         203196  82
   25-29         289679  89
   30-34         303177  86
   35-39         364289 118
   40-44         377476 104
   45-49         361264  65
   50-54         451343  49
   55-59         461593  35
   60-64         203724  15
